# Pipeline

In [0]:
# ============================================================
# Librerias
# ============================================================
import os
import shutil
import json
import logging
import re
import subprocess
import time
import uuid
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

In [0]:
# ============================================================
# Parámetros
# ============================================================
CATALOG_NAME = "nyc_taxi_sebastian"

TAXI_MONTH = "2023-01"
YELLOW_TAXI_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
TAXI_ZONE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
MIN_TRIPS_ZONE_RANKING = 100

RAW_SCHEMA = f"{CATALOG_NAME}.raw"
TRUSTED_SCHEMA = f"{CATALOG_NAME}.trusted"
REFINED_SCHEMA = f"{CATALOG_NAME}.refined"

RAW_TRIPS_TABLE = f"{RAW_SCHEMA}.yellow_taxi_trips"
RAW_ZONES_TABLE = f"{RAW_SCHEMA}.taxi_zones"
TRUSTED_TRIPS_TABLE = f"{TRUSTED_SCHEMA}.yellow_taxi_trips_enriched"

KPI_TEMPORAL_TABLE = f"{REFINED_SCHEMA}.kpi_temporal_demand"
KPI_TEMPORAL_PEAKS_TABLE = f"{REFINED_SCHEMA}.kpi_temporal_peaks"
KPI_ZONE_EFF_TABLE = f"{REFINED_SCHEMA}.kpi_zone_economic_efficiency"
KPI_TOP10_ZONES_TABLE = f"{REFINED_SCHEMA}.kpi_top10_profitable_zones"
KPI_BOROUGH_EFF_TABLE = f"{REFINED_SCHEMA}.kpi_borough_economic_efficiency"
DQ_REPORT_TABLE = f"{REFINED_SCHEMA}.data_quality_report"
DQ_IMPACT_TABLE = f"{REFINED_SCHEMA}.kpi_data_quality_impact"
EXECUTION_REPORT_TABLE = f"{REFINED_SCHEMA}.execution_report"

# Unity Catalog Volume para archivos fuente.
VOLUME_FULL_NAME = f"{RAW_SCHEMA}.nyc_taxi_files"
VOLUME_BASE_DIR = f"/Volumes/{CATALOG_NAME}/raw/nyc_taxi_files"
VOLUME_RAW_FILES_DIR = f"{VOLUME_BASE_DIR}/raw_files"
VOLUME_REPORTS_DIR = f"{VOLUME_BASE_DIR}/reports"

YELLOW_TAXI_VOLUME_PATH = f"{VOLUME_RAW_FILES_DIR}/yellow_tripdata_{TAXI_MONTH}.parquet"
TAXI_ZONE_CSV_PATH = f"{VOLUME_BASE_DIR}/taxi_zone_lookup.csv"

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().isoformat()

/home/spark-e20b2a94-4d81-4d93-9677-9d/.ipykernel/7993/command-6593938700814467-1449281720:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TS = datetime.utcnow().isoformat()


In [0]:
# ============================================================
# Se agrega el logger para registrar mensajes de INFO, WARNING y ERROR
# durante cada etapa del proceso.
# ============================================================
logger = logging.getLogger("nyc_taxi_medallion")
logger.setLevel(logging.INFO)

if not logger.handlers:
    stream_handler = logging.StreamHandler()
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

stage_metrics = {}

execution_report = {
    "run_id": RUN_ID,
    "run_timestamp_utc": RUN_TS,
    "catalog": CATALOG_NAME,
    "taxi_month": TAXI_MONTH,
    "source_yellow_taxi_url": YELLOW_TAXI_URL,
    "source_taxi_zone_url": TAXI_ZONE_URL,
    "stage_metrics": {},
    "tables": {},
    "kpis": {}
}

In [0]:
# ============================================================
# Se definen funciones reutilizables para evitar repetir lógica
# en las diferentes etapas del proceso.
# ============================================================
def run_with_retry(stage_name, func, retries=2, sleep_seconds=5):
    """
    Ejecuta una etapa con reintentos, logs y medición de tiempo.
    """
    start = time.time()
    last_error = None

    for attempt in range(1, retries + 2):
        try:
            logger.info(f"Starting stage={stage_name}, attempt={attempt}")
            result = func()
            elapsed = round(time.time() - start, 2)

            stage_metrics[stage_name] = {
                "status": "SUCCESS",
                "elapsed_seconds": elapsed,
                "attempts": attempt
            }

            logger.info(f"Finished stage={stage_name}, elapsed_seconds={elapsed}")
            return result

        except Exception as exc:
            last_error = exc
            logger.error(f"Stage={stage_name} failed on attempt={attempt}: {str(exc)}")

            if attempt <= retries:
                wait = sleep_seconds * attempt
                logger.warning(f"Retrying stage={stage_name} in {wait} seconds")
                time.sleep(wait)

    elapsed = round(time.time() - start, 2)

    stage_metrics[stage_name] = {
        "status": "FAILED",
        "elapsed_seconds": elapsed,
        "attempts": retries + 1,
        "error": str(last_error)
    }

    raise last_error


def snake_case(name):
    """
    Convierte nombres de columnas a snake_case.
    Ejemplo: PULocationID -> pulocation_id
    """
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    return name.lower().strip("_")


def standardize_columns(df):
    """
    Estandariza todas las columnas de un DataFrame a snake_case.
    """
    for c in df.columns:
        df = df.withColumnRenamed(c, snake_case(c))
    return df


def count_df(df):
    """
    Conteo explícito para logs y métricas.
    """
    return df.count()


def write_delta_table(df, table_name, mode="overwrite", partition_cols=None, comment=None):
    """
    Escribe un DataFrame como tabla Delta administrada en Unity Catalog.
    """
    writer = (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
    )

    if partition_cols:
        writer = writer.partitionBy(*partition_cols)

    writer.saveAsTable(table_name)

    if comment:
        try:
            safe_comment = comment.replace("'", "''")
            spark.sql(f"COMMENT ON TABLE {table_name} IS '{safe_comment}'")
        except Exception as exc:
            logger.warning(f"Could not set table comment for {table_name}: {str(exc)}")


def safe_table_count(table_name):
    """
    Retorna conteo de una tabla si existe.
    """
    try:
        return spark.table(table_name).count()
    except Exception:
        return None

In [0]:
# ============================================================
# Configuración del Unity Catalog: schemas y Volume
# Se crean los tres schemas que representan las capas de la arquitectura Medallion
# También se crea un Unity Catalog Volume para almacenar archivos no tabulares
# utilizados por el pipeline, como el Parquet fuente, el CSV de zonas y el
# reporte JSON de ejecución.
# ============================================================
def setup_catalog_and_schemas():
    """
    Crea schemas Raw, Trusted y Refined dentro del catálogo workspace.
    También crea un Unity Catalog Volume para archivos fuente y reportes.
    """
    spark.sql(f"USE CATALOG {CATALOG_NAME}")

    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS {RAW_SCHEMA}
        COMMENT 'Raw layer: source data with minimal transformation'
        """
    )

    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS {TRUSTED_SCHEMA}
        COMMENT 'Trusted layer: cleaned, validated and enriched data'
        """
    )

    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS {REFINED_SCHEMA}
        COMMENT 'Refined layer: KPIs and analytical outputs'
        """
    )

    spark.sql(
        f"""
        CREATE VOLUME IF NOT EXISTS {VOLUME_FULL_NAME}
        COMMENT 'Volume for NYC taxi source files and execution reports'
        """
    )

    # Crear carpetas internas dentro del Volume.
    os.makedirs(VOLUME_RAW_FILES_DIR, exist_ok=True)
    os.makedirs(VOLUME_REPORTS_DIR, exist_ok=True)

    logger.info(f"Catalog, schemas and volume ready: {CATALOG_NAME}")
    logger.info(f"Volume path: {VOLUME_BASE_DIR}")


run_with_retry("setup_catalog_and_schemas", setup_catalog_and_schemas)

2026-04-30 02:59:45,342 | INFO | Starting stage=setup_catalog_and_schemas, attempt=1
2026-04-30 02:59:50,144 | INFO | Catalog, schemas and volume ready: db_pruebatecnica
2026-04-30 02:59:50,145 | INFO | Volume path: /Volumes/db_pruebatecnica/raw/nyc_taxi_files
2026-04-30 02:59:50,146 | INFO | Finished stage=setup_catalog_and_schemas, elapsed_seconds=4.8


In [0]:
# ============================================================
# Descarga de archivos fuente hacia Unity Catalog
# ============================================================
def download_file_to_volume(source_url, target_path, local_path):
    """
    Descarga un archivo público con curl y lo copia a Unity Catalog Volume.
    """
    logger.info(f"Downloading file from {source_url}")

    cmd = ["bash", "-lc", f"curl -L --fail --retry 3 -o {local_path} {source_url}"]
    subprocess.run(cmd, check=True)

    shutil.copyfile(local_path, target_path)

    logger.info(f"File copied to {target_path}")
    return target_path


def download_yellow_taxi_parquet():
    os.makedirs(VOLUME_RAW_FILES_DIR, exist_ok=True)

    file_name = f"yellow_tripdata_{TAXI_MONTH}.parquet"
    local_path = f"/tmp/{file_name}"

    return download_file_to_volume(
        source_url=YELLOW_TAXI_URL,
        target_path=YELLOW_TAXI_VOLUME_PATH,
        local_path=local_path
    )


def download_taxi_zone_lookup():
    os.makedirs(VOLUME_BASE_DIR, exist_ok=True)

    local_path = "/tmp/taxi_zone_lookup.csv"

    return download_file_to_volume(
        source_url=TAXI_ZONE_URL,
        target_path=TAXI_ZONE_CSV_PATH,
        local_path=local_path
    )


yellow_taxi_path = run_with_retry("download_yellow_taxi_parquet", download_yellow_taxi_parquet)
taxi_zone_path = run_with_retry("download_taxi_zone_lookup", download_taxi_zone_lookup)

logger.info(f"Yellow taxi path ready: {yellow_taxi_path}")
logger.info(f"Taxi zone CSV path ready: {taxi_zone_path}")

2026-04-30 02:59:50,369 | INFO | Starting stage=download_yellow_taxi_parquet, attempt=1
2026-04-30 02:59:50,521 | INFO | Downloading file from https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 45.4M  100 45.4M    0     0  51.3M      0 --:--:-- --:--:-- --:--:-- 51.3M
2026-04-30 02:59:52,349 | INFO | File copied to /Volumes/db_pruebatecnica/raw/nyc_taxi_files/raw_files/yellow_tripdata_2023-01.parquet
2026-04-30 02:59:52,350 | INFO | Finished stage=download_yellow_taxi_parquet, elapsed_seconds=1.98
2026-04-30 02:59:52,351 | INFO | Starting stage=download_taxi_zone_lookup, attempt=1
2026-04-30 02:59:52,357 | INFO | Downloading file from https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                      

## Capa Raw

In [0]:
# ============================================================
# Raw
# ============================================================
def ingest_raw():
    logger.info("Reading yellow taxi parquet")

    trips_raw = spark.read.parquet(yellow_taxi_path)
    trips_raw = standardize_columns(trips_raw)

    # Renombres de negocio para mayor claridad.
    rename_map = {
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "pu_location_id": "pickup_location_id",
        "do_location_id": "dropoff_location_id",
        "pulocation_id": "pickup_location_id",
        "dolocation_id": "dropoff_location_id"
    }

    for old_col, new_col in rename_map.items():
        if old_col in trips_raw.columns:
            trips_raw = trips_raw.withColumnRenamed(old_col, new_col)

    trips_raw = (
        trips_raw
        .withColumn("source_file", F.lit(YELLOW_TAXI_URL))
        .withColumn("taxi_month", F.lit(TAXI_MONTH))
        .withColumn("ingestion_timestamp", F.current_timestamp())
    )

    logger.info("Reading taxi zone CSV")

    zones_raw = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(taxi_zone_path)
    )

    zones_raw = standardize_columns(zones_raw)

    # En el CSV público normalmente viene LocationID.
    if "location_id" not in zones_raw.columns and "locationid" in zones_raw.columns:
        zones_raw = zones_raw.withColumnRenamed("locationid", "location_id")

    zones_raw = (
        zones_raw
        .withColumn("source_file", F.lit(TAXI_ZONE_URL))
        .withColumn("ingestion_timestamp", F.current_timestamp())
    )

    raw_trip_count = count_df(trips_raw)
    raw_zone_count = count_df(zones_raw)

    logger.info(f"Raw trips read: {raw_trip_count}")
    logger.info(f"Raw zones read: {raw_zone_count}")

    write_delta_table(
        trips_raw,
        RAW_TRIPS_TABLE,
        mode="overwrite",
        comment="Raw NYC Yellow Taxi trips for selected month"
    )

    write_delta_table(
        zones_raw,
        RAW_ZONES_TABLE,
        mode="overwrite",
        comment="Raw NYC Taxi Zone Lookup"
    )

    execution_report["tables"]["raw_trips"] = {
        "table": RAW_TRIPS_TABLE,
        "rows": raw_trip_count
    }

    execution_report["tables"]["raw_zones"] = {
        "table": RAW_ZONES_TABLE,
        "rows": raw_zone_count
    }

    return raw_trip_count, raw_zone_count


raw_trip_count, raw_zone_count = run_with_retry("ingest_raw", ingest_raw)

2026-04-30 02:59:52,858 | INFO | Starting stage=ingest_raw, attempt=1
2026-04-30 02:59:52,859 | INFO | Reading yellow taxi parquet
2026-04-30 02:59:57,812 | INFO | Reading taxi zone CSV
2026-04-30 03:00:04,337 | INFO | Raw trips read: 3066766
2026-04-30 03:00:04,339 | INFO | Raw zones read: 265
2026-04-30 03:00:25,643 | INFO | Finished stage=ingest_raw, elapsed_seconds=32.78


## Capa Trusted

In [0]:
# ============================================================
# Trusted: se hace validación, limpieza, y manejo de valores atípicos.
# ============================================================
def build_quality_report_and_trusted():
    trips = spark.table(RAW_TRIPS_TABLE)
    zones = spark.table(RAW_ZONES_TABLE)

    # Tipado defensivo.
    trips = (
        trips
        .withColumn("pickup_datetime", F.col("pickup_datetime").cast("timestamp"))
        .withColumn("dropoff_datetime", F.col("dropoff_datetime").cast("timestamp"))
        .withColumn("trip_distance", F.col("trip_distance").cast("double"))
        .withColumn("fare_amount", F.col("fare_amount").cast("double"))
        .withColumn("total_amount", F.col("total_amount").cast("double"))
        .withColumn("passenger_count", F.col("passenger_count").cast("double"))
        .withColumn("pickup_location_id", F.col("pickup_location_id").cast("int"))
        .withColumn("dropoff_location_id", F.col("dropoff_location_id").cast("int"))
    )

    trips_base = (
        trips
        .withColumn(
            "trip_duration_minutes",
            (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / F.lit(60.0)
        )
        .withColumn("trip_duration_hours", F.col("trip_duration_minutes") / F.lit(60.0))
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
    )

    raw_count = count_df(trips_base)

    raw_total_revenue = (
        trips_base
        .agg(F.sum(F.coalesce(F.col("total_amount"), F.lit(0.0))).alias("value"))
        .first()["value"]
    ) or 0.0

    # Reglas de calidad.
    rules = [
        (
            "pickup_before_dropoff",
            "pickup_datetime debe ser menor que dropoff_datetime",
            F.col("pickup_datetime").isNotNull()
            & F.col("dropoff_datetime").isNotNull()
            & (F.col("pickup_datetime") < F.col("dropoff_datetime"))
        ),
        (
            "trip_distance_positive",
            "trip_distance debe ser mayor que 0",
            F.col("trip_distance").isNotNull()
            & (F.col("trip_distance") > 0)
        ),
        (
            "fare_amount_positive",
            "fare_amount debe ser mayor que 0",
            F.col("fare_amount").isNotNull()
            & (F.col("fare_amount") > 0)
        ),
        (
            "critical_fields_not_null",
            "Campos críticos no deben ser nulos",
            F.col("pickup_datetime").isNotNull()
            & F.col("dropoff_datetime").isNotNull()
            & F.col("pickup_location_id").isNotNull()
            & F.col("trip_distance").isNotNull()
            & F.col("fare_amount").isNotNull()
            & F.col("total_amount").isNotNull()
        ),
        (
            "duration_not_extreme",
            "Duración entre 1 y 180 minutos",
            F.col("trip_duration_minutes").isNotNull()
            & (F.col("trip_duration_minutes") >= 1)
            & (F.col("trip_duration_minutes") <= 180)
        ),
        (
            "distance_not_extreme",
            "Distancia menor o igual a 100 millas",
            F.col("trip_distance").isNotNull()
            & (F.col("trip_distance") <= 100)
        ),
        (
            "total_amount_not_extreme",
            "total_amount entre 0 y 1000 USD",
            F.col("total_amount").isNotNull()
            & (F.col("total_amount") >= 0)
            & (F.col("total_amount") <= 1000)
        )
    ]

    dq_rows = []
    valid_condition = F.lit(True)

    for rule_name, rule_description, condition in rules:
        pass_count = trips_base.filter(condition).count()
        fail_df = trips_base.filter(~condition)
        fail_count = fail_df.count()

        failed_revenue = (
            fail_df
            .agg(F.sum(F.coalesce(F.col("total_amount"), F.lit(0.0))).alias("value"))
            .first()["value"]
        ) or 0.0

        dq_rows.append({
            "run_id": RUN_ID,
            "run_timestamp_utc": RUN_TS,
            "rule_name": rule_name,
            "rule_description": rule_description,
            "total_records": raw_count,
            "passed_records": pass_count,
            "failed_records": fail_count,
            "failed_percentage": round((fail_count / raw_count) * 100, 4) if raw_count else 0.0,
            "failed_revenue_amount": float(failed_revenue)
        })

        valid_condition = valid_condition & condition

    dq_schema = T.StructType([
        T.StructField("run_id", T.StringType(), False),
        T.StructField("run_timestamp_utc", T.StringType(), False),
        T.StructField("rule_name", T.StringType(), False),
        T.StructField("rule_description", T.StringType(), False),
        T.StructField("total_records", T.LongType(), False),
        T.StructField("passed_records", T.LongType(), False),
        T.StructField("failed_records", T.LongType(), False),
        T.StructField("failed_percentage", T.DoubleType(), False),
        T.StructField("failed_revenue_amount", T.DoubleType(), False),
    ])

    dq_report = spark.createDataFrame(dq_rows, dq_schema)

    write_delta_table(
        dq_report,
        DQ_REPORT_TABLE,
        mode="overwrite",
        comment="Data quality report with pass and fail metrics by validation rule"
    )

    logger.info(f"Data quality report written: {DQ_REPORT_TABLE}")

    zones_clean = (
        zones
        .select(
            F.col("location_id").cast("int").alias("location_id"),
            F.trim(F.col("borough")).alias("borough"),
            F.trim(F.col("zone")).alias("zone"),
            F.trim(F.col("service_zone")).alias("service_zone")
        )
        .dropDuplicates(["location_id"])
    )

    trusted = (
        trips_base
        .filter(valid_condition)
        .withColumn("passenger_count", F.coalesce(F.col("passenger_count"), F.lit(0.0)))
        .join(
            F.broadcast(zones_clean),
            F.col("pickup_location_id") == F.col("location_id"),
            "left"
        )
        .drop("location_id")
        .withColumnRenamed("borough", "pickup_borough")
        .withColumnRenamed("zone", "pickup_zone")
        .withColumnRenamed("service_zone", "pickup_service_zone")
        .withColumn("pickup_borough", F.coalesce(F.col("pickup_borough"), F.lit("Unknown")))
        .withColumn("pickup_zone", F.coalesce(F.col("pickup_zone"), F.lit("Unknown")))
        .withColumn("pickup_service_zone", F.coalesce(F.col("pickup_service_zone"), F.lit("Unknown")))
        .withColumn("revenue_per_mile", F.col("total_amount") / F.col("trip_distance"))
        .withColumn("speed_mph", F.col("trip_distance") / F.col("trip_duration_hours"))
        .withColumn("trusted_load_timestamp", F.current_timestamp())
    )

    trusted_count = count_df(trusted)

    trusted_total_revenue = (
        trusted
        .agg(F.sum("total_amount").alias("value"))
        .first()["value"]
    ) or 0.0

    logger.info(f"Trusted trips written: {trusted_count}")
    logger.warning(f"Discarded records after quality rules: {raw_count - trusted_count}")

    write_delta_table(
        trusted,
        TRUSTED_TRIPS_TABLE,
        mode="overwrite",
        partition_cols=["pickup_date"],
        comment="Trusted NYC Yellow Taxi trips cleaned and enriched with pickup zone"
    )

    dq_impact = spark.createDataFrame([{
        "run_id": RUN_ID,
        "run_timestamp_utc": RUN_TS,
        "raw_records": raw_count,
        "trusted_records": trusted_count,
        "discarded_records": raw_count - trusted_count,
        "discarded_records_percentage": round(((raw_count - trusted_count) / raw_count) * 100, 4) if raw_count else 0.0,
        "raw_total_revenue": float(raw_total_revenue),
        "trusted_total_revenue": float(trusted_total_revenue),
        "discarded_revenue": float(raw_total_revenue - trusted_total_revenue),
        "discarded_revenue_percentage": round(((raw_total_revenue - trusted_total_revenue) / raw_total_revenue) * 100, 4) if raw_total_revenue else 0.0
    }])

    write_delta_table(
        dq_impact,
        DQ_IMPACT_TABLE,
        mode="overwrite",
        comment="Impact of data quality filters on record count and total revenue"
    )

    execution_report["tables"]["trusted_trips"] = {
        "table": TRUSTED_TRIPS_TABLE,
        "rows": trusted_count
    }

    execution_report["tables"]["data_quality_report"] = {
        "table": DQ_REPORT_TABLE,
        "rows": dq_report.count()
    }

    execution_report["tables"]["data_quality_impact"] = {
        "table": DQ_IMPACT_TABLE,
        "rows": dq_impact.count()
    }

    execution_report["quality_summary"] = {
        "raw_records": raw_count,
        "trusted_records": trusted_count,
        "discarded_records": raw_count - trusted_count,
        "raw_total_revenue": float(raw_total_revenue),
        "trusted_total_revenue": float(trusted_total_revenue),
        "discarded_revenue": float(raw_total_revenue - trusted_total_revenue)
    }

    return trusted_count


trusted_count = run_with_retry("build_quality_report_and_trusted", build_quality_report_and_trusted)

2026-04-30 03:00:26,019 | INFO | Starting stage=build_quality_report_and_trusted, attempt=1
2026-04-30 03:00:50,194 | INFO | Data quality report written: db_pruebatecnica.refined.data_quality_report
2026-04-30 03:00:53,268 | INFO | Trusted trips written: 2986725
2026-04-30 03:00:53,270 | WARNING | Discarded records after quality rules: 80041
2026-04-30 03:01:20,932 | INFO | Finished stage=build_quality_report_and_trusted, elapsed_seconds=54.91


## Capa Refined y KPIs

In [0]:
# ============================================================
# Refined: KPIs
# KPIs construidos:
# - Patrón de demanda temporal: viajes, duración promedio y tarifa promedio
#   por día de la semana y franja horaria.
# - Picos de demanda temporal: top de combinaciones día/franja con más viajes.
# - Eficiencia económica por zona: ingreso promedio por milla, velocidad
#   promedio, ingreso total y ranking de zonas.
# - Top 10 zonas más rentables.
# - Eficiencia económica agregada por borough.
# ============================================================
def build_refined_kpis():
    trusted = spark.table(TRUSTED_TRIPS_TABLE)

    h = F.hour("pickup_datetime")

    trusted_kpi = (
        trusted
        .withColumn("pickup_hour", h)
        .withColumn(
            "time_band",
            F.when((h >= 0) & (h <= 3), F.lit("00-03"))
             .when((h >= 4) & (h <= 7), F.lit("04-07"))
             .when((h >= 8) & (h <= 11), F.lit("08-11"))
             .when((h >= 12) & (h <= 15), F.lit("12-15"))
             .when((h >= 16) & (h <= 19), F.lit("16-19"))
             .otherwise(F.lit("20-23"))
        )
        .withColumn("day_of_week_num", F.dayofweek("pickup_datetime"))
        .withColumn(
            "day_of_week",
            F.when(F.col("day_of_week_num") == 1, F.lit("Sunday"))
             .when(F.col("day_of_week_num") == 2, F.lit("Monday"))
             .when(F.col("day_of_week_num") == 3, F.lit("Tuesday"))
             .when(F.col("day_of_week_num") == 4, F.lit("Wednesday"))
             .when(F.col("day_of_week_num") == 5, F.lit("Thursday"))
             .when(F.col("day_of_week_num") == 6, F.lit("Friday"))
             .when(F.col("day_of_week_num") == 7, F.lit("Saturday"))
        )
    )

    # KPI 1: Patrón de demanda temporal.
    kpi_temporal = (
        trusted_kpi
        .groupBy("day_of_week_num", "day_of_week", "time_band")
        .agg(
            F.count("*").alias("trip_count"),
            F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
            F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount"),
            F.round(F.avg("total_amount"), 2).alias("avg_total_amount")
        )
        .orderBy("day_of_week_num", "time_band")
    )

    write_delta_table(
        kpi_temporal,
        KPI_TEMPORAL_TABLE,
        mode="overwrite",
        comment="Temporal demand KPI by weekday and six time bands"
    )

    kpi_temporal_peaks = (
        kpi_temporal
        .orderBy(F.desc("trip_count"))
        .limit(10)
    )

    write_delta_table(
        kpi_temporal_peaks,
        KPI_TEMPORAL_PEAKS_TABLE,
        mode="overwrite",
        comment="Top temporal demand peaks"
    )

    # KPI 2: Eficiencia económica por zona.
    kpi_zone_efficiency = (
        trusted_kpi
        .groupBy("pickup_borough", "pickup_zone")
        .agg(
            F.count("*").alias("trip_count"),
            F.round(F.sum("total_amount"), 2).alias("total_revenue"),
            F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
            F.round(F.avg("revenue_per_mile"), 2).alias("avg_revenue_per_mile"),
            F.round(F.avg("speed_mph"), 2).alias("avg_speed_mph"),
            F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
            F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes")
        )
        .withColumn(
            "profitability_rank",
            F.dense_rank().over(
                Window.orderBy(F.desc("avg_revenue_per_mile"), F.desc("total_revenue"))
            )
        )
    )

    write_delta_table(
        kpi_zone_efficiency,
        KPI_ZONE_EFF_TABLE,
        mode="overwrite",
        comment="Economic efficiency KPI by pickup borough and zone"
    )

    top10_zones = (
        kpi_zone_efficiency
        .filter(F.col("trip_count") >= MIN_TRIPS_ZONE_RANKING)
        .orderBy(F.desc("avg_revenue_per_mile"), F.desc("total_revenue"))
        .limit(10)
    )

    write_delta_table(
        top10_zones,
        KPI_TOP10_ZONES_TABLE,
        mode="overwrite",
        comment="Top 10 most profitable pickup zones using average revenue per mile"
    )

    kpi_borough_efficiency = (
        trusted_kpi
        .groupBy("pickup_borough")
        .agg(
            F.count("*").alias("trip_count"),
            F.round(F.sum("total_amount"), 2).alias("total_revenue"),
            F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
            F.round(F.avg("revenue_per_mile"), 2).alias("avg_revenue_per_mile"),
            F.round(F.avg("speed_mph"), 2).alias("avg_speed_mph")
        )
        .orderBy(F.desc("avg_revenue_per_mile"))
    )

    write_delta_table(
        kpi_borough_efficiency,
        KPI_BOROUGH_EFF_TABLE,
        mode="overwrite",
        comment="Economic efficiency KPI by pickup borough"
    )

    peak_rows = [row.asDict() for row in kpi_temporal_peaks.collect()]
    top_zone_rows = [row.asDict() for row in top10_zones.collect()]

    execution_report["tables"]["kpi_temporal_demand"] = {
        "table": KPI_TEMPORAL_TABLE,
        "rows": kpi_temporal.count()
    }

    execution_report["tables"]["kpi_temporal_peaks"] = {
        "table": KPI_TEMPORAL_PEAKS_TABLE,
        "rows": len(peak_rows)
    }

    execution_report["tables"]["kpi_zone_economic_efficiency"] = {
        "table": KPI_ZONE_EFF_TABLE,
        "rows": kpi_zone_efficiency.count()
    }

    execution_report["tables"]["kpi_top10_profitable_zones"] = {
        "table": KPI_TOP10_ZONES_TABLE,
        "rows": len(top_zone_rows)
    }

    execution_report["tables"]["kpi_borough_economic_efficiency"] = {
        "table": KPI_BOROUGH_EFF_TABLE,
        "rows": kpi_borough_efficiency.count()
    }

    execution_report["kpis"]["temporal_peaks_top10"] = peak_rows
    execution_report["kpis"]["top10_profitable_zones"] = top_zone_rows

    return True


run_with_retry("build_refined_kpis", build_refined_kpis)

2026-04-30 03:01:21,298 | INFO | Starting stage=build_refined_kpis, attempt=1
/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
2026-04-30 03:01:50,178 | INFO | Finished stage=build_refined_kpis, elapsed_seconds=28.88


True

## Reporte final de ejecución

In [0]:
# ============================================================
# se consolida la información de ejecución en un archivo JSON
# ============================================================
def write_execution_report():
    os.makedirs(VOLUME_REPORTS_DIR, exist_ok=True)

    execution_report["stage_metrics"] = stage_metrics
    execution_report["final_status"] = "SUCCESS"
    execution_report["finished_timestamp_utc"] = datetime.utcnow().isoformat()

    report_json = json.dumps(execution_report, indent=2, default=str)
    report_path = f"{VOLUME_REPORTS_DIR}/execution_report_{RUN_ID}.json"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_json)

    report_df = spark.createDataFrame([{
        "run_id": RUN_ID,
        "run_timestamp_utc": RUN_TS,
        "finished_timestamp_utc": execution_report["finished_timestamp_utc"],
        "catalog": CATALOG_NAME,
        "taxi_month": TAXI_MONTH,
        "status": "SUCCESS",
        "raw_records": execution_report.get("quality_summary", {}).get("raw_records"),
        "trusted_records": execution_report.get("quality_summary", {}).get("trusted_records"),
        "discarded_records": execution_report.get("quality_summary", {}).get("discarded_records"),
        "report_json_path": report_path,
        "report_json": report_json
    }])

    write_delta_table(
        report_df,
        EXECUTION_REPORT_TABLE,
        mode="overwrite",
        comment="Final execution report for NYC taxi Medallion pipeline"
    )

    logger.info(f"Execution report written to {report_path}")
    logger.info(f"Execution report table written to {EXECUTION_REPORT_TABLE}")

    return report_path


report_path = run_with_retry("write_execution_report", write_execution_report)

2026-04-30 03:01:50,484 | INFO | Starting stage=write_execution_report, attempt=1
/home/spark-e20b2a94-4d81-4d93-9677-9d/.ipykernel/7993/command-6593938700814479-2470055224:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  execution_report["finished_timestamp_utc"] = datetime.utcnow().isoformat()
2026-04-30 03:01:54,689 | INFO | Execution report written to /Volumes/db_pruebatecnica/raw/nyc_taxi_files/reports/execution_report_139925ad-ecc9-49a0-8d38-8679529c504c.json
2026-04-30 03:01:54,690 | INFO | Execution report table written to db_pruebatecnica.refined.execution_report
2026-04-30 03:01:54,691 | INFO | Finished stage=write_execution_report, elapsed_seconds=4.21


In [0]:
display(spark.table(DQ_REPORT_TABLE))
display(spark.table(DQ_IMPACT_TABLE))
display(spark.table(KPI_TEMPORAL_PEAKS_TABLE))
display(spark.table(KPI_TOP10_ZONES_TABLE))
display(spark.table(EXECUTION_REPORT_TABLE))

print(f"Execution report JSON: {report_path}")

run_id,run_timestamp_utc,rule_name,rule_description,total_records,passed_records,failed_records,failed_percentage,failed_revenue_amount
139925ad-ecc9-49a0-8d38-8679529c504c,2026-04-30T02:59:43.902561,pickup_before_dropoff,pickup_datetime debe ser menor que dropoff_datetime,3066766,3065645,1121,0.0366,27652.129999999997
139925ad-ecc9-49a0-8d38-8679529c504c,2026-04-30T02:59:43.902561,trip_distance_positive,trip_distance debe ser mayor que 0,3066766,3020904,45862,1.4955,1361646.5899999992
139925ad-ecc9-49a0-8d38-8679529c504c,2026-04-30T02:59:43.902561,fare_amount_positive,fare_amount debe ser mayor que 0,3066766,3040607,26159,0.853,-602987.9000000026
139925ad-ecc9-49a0-8d38-8679529c504c,2026-04-30T02:59:43.902561,critical_fields_not_null,Campos críticos no deben ser nulos,3066766,3066766,0,0.0,0.0
139925ad-ecc9-49a0-8d38-8679529c504c,2026-04-30T02:59:43.902561,duration_not_extreme,Duración entre 1 y 180 minutos,3066766,3030383,36383,1.1864,1100587.9999999972
139925ad-ecc9-49a0-8d38-8679529c504c,2026-04-30T02:59:43.902561,distance_not_extreme,Distancia menor o igual a 100 millas,3066766,3066678,88,0.0029,10761.959999999997
139925ad-ecc9-49a0-8d38-8679529c504c,2026-04-30T02:59:43.902561,total_amount_not_extreme,total_amount entre 0 y 1000 USD,3066766,3041561,25205,0.8219,-603827.4400000026


discarded_records,discarded_records_percentage,discarded_revenue,discarded_revenue_percentage,raw_records,raw_total_revenue,run_id,run_timestamp_utc,trusted_records,trusted_total_revenue
80041,2.6099,1244918.0199904293,1.5023,3066766,8.28651922197824E7,139925ad-ecc9-49a0-8d38-8679529c504c,2026-04-30T02:59:43.902561,2986725,8.162027419979197E7


day_of_week_num,day_of_week,time_band,trip_count,avg_duration_minutes,avg_fare_amount,avg_total_amount
3,Tuesday,16-19,131030,14.63,17.65,28.34
5,Thursday,16-19,118416,15.92,17.89,28.58
4,Wednesday,16-19,116136,14.72,17.26,27.82
3,Tuesday,12-15,115320,16.51,19.01,27.1
6,Friday,16-19,113490,15.07,17.62,28.28
2,Monday,16-19,108719,14.39,18.8,28.96
7,Saturday,16-19,108130,15.47,17.87,25.53
1,Sunday,12-15,104864,14.61,19.67,28.09
2,Monday,12-15,103251,15.16,19.22,27.5
3,Tuesday,08-11,101251,15.61,17.79,25.66


pickup_borough,pickup_zone,trip_count,total_revenue,avg_total_amount,avg_revenue_per_mile,avg_speed_mph,avg_trip_distance,avg_duration_minutes,profitability_rank
N/A,Outside of NYC,334,25397.66,76.04,49.28,39.55,8.62,19.66,4
Queens,Jackson Heights,431,12973.97,30.1,31.46,15.87,4.23,15.51,8
Brooklyn,South Williamsburg,104,3173.02,30.51,22.13,13.61,4.27,19.5,14
Queens,Kew Gardens Hills,116,3359.36,28.96,18.16,14.33,5.26,19.74,17
Queens,Elmhurst/Maspeth,103,3241.33,31.47,17.55,16.28,4.48,17.13,19
Brooklyn,Crown Heights North,420,12547.34,29.87,17.23,10.09,4.77,27.84,21
Queens,Queensbridge/Ravenswood,423,10076.81,23.82,16.59,12.26,3.37,16.72,25
Brooklyn,Red Hook,101,4767.38,47.2,16.42,16.67,7.75,28.24,28
Queens,Long Island City/Hunters Point,926,43550.82,47.03,15.31,17.41,4.59,17.08,31
Queens,Maspeth,209,12775.86,61.13,14.72,23.35,6.18,18.86,32


catalog discarded_records finished_timestamp_utc raw_records report_json report_json_path run_id run_timestamp_utc status taxi_month trusted_records db_pruebatecnica 80041 2026-04-30T03:01:50.778843 3066766 {
 "run_id": "139925ad-ecc9-49a0-8d38-8679529c504c",
 "run_timestamp_utc": "2026-04-30T02:59:43.902561",
 "catalog": "db_pruebatecnica",
 "taxi_month": "2023-01",
 "source_yellow_taxi_url": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet",
 "source_taxi_zone_url": "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
 "stage_metrics": {
 "setup_catalog_and_schemas": {
 "status": "SUCCESS",
 "elapsed_seconds": 4.8,
 "attempts": 1
 },
 "download_yellow_taxi_parquet": {
 "status": "SUCCESS",
 "elapsed_seconds": 1.98,
 "attempts": 1
 },
 "download_taxi_zone_lookup": {
 "status": "SUCCESS",
 "elapsed_seconds": 0.22,
 "attempts": 1
 },
 "ingest_raw": {
 "status": "SUCCESS",
 "elapsed_seconds": 32.78,
 "attempts": 1
 },
 "build_quality_report_and_trusted": {
 "status": "SUCCESS",
 "elapsed_seconds": 54.91,
 "attempts": 1
 },
 "build_refined_kpis": {
 "status": "SUCCESS",
 "elapsed_seconds": 28.88,
 "attempts": 1
 }
 },
 "tables": {
 "raw_trips": {
 "table": "db_pruebatecnica.raw.yellow_taxi_trips",
 "rows": 3066766
 },
 "raw_zones": {
 "table": "db_pruebatecnica.raw.taxi_zones",
 "rows": 265
 },
 "trusted_trips": {
 "table": "db_pruebatecnica.trusted.yellow_taxi_trips_enriched",
 "rows": 2986725
 },
 "data_quality_report": {
 "table": "db_pruebatecnica.refined.data_quality_report",
 "rows": 7
 },
 "data_quality_impact": {
 "table": "db_pruebatecnica.refined.kpi_data_quality_impact",
 "rows": 1
 },
 "kpi_temporal_demand": {
 "table": "db_pruebatecnica.refined.kpi_temporal_demand",
 "rows": 42
 },
 "kpi_temporal_peaks": {
 "table": "db_pruebatecnica.refined.kpi_temporal_peaks",
 "rows": 10
 },
 "kpi_zone_economic_efficiency": {
 "table": "db_pruebatecnica.refined.kpi_zone_economic_efficiency",
 "rows": 253
 },
 "kpi_top10_profitable_zones": {
 "table": "db_pruebatecnica.refined.kpi_top10_profitable_zones",
 "rows": 10
 },
 "kpi_borough_economic_efficiency": {
 "table": "db_pruebatecnica.refined.kpi_borough_economic_efficiency",
 "rows": 8
 }
 },
 "kpis": {
 "temporal_peaks_top10": [
 {
 "day_of_week_num": 3,
 "day_of_week": "Tuesday",
 "time_band": "16-19",
 "trip_count": 131030,
 "avg_duration_minutes": 14.63,
 "avg_fare_amount": 17.65,
 "avg_total_amount": 28.34
 },
 {
 "day_of_week_num": 5,
 "day_of_week": "Thursday",
 "time_band": "16-19",
 "trip_count": 118416,
 "avg_duration_minutes": 15.92,
 "avg_fare_amount": 17.89,
 "avg_total_amount": 28.58
 },
 {
 "day_of_week_num": 4,
 "day_of_week": "Wednesday",
 "time_band": "16-19",
 "trip_count": 116136,
 "avg_duration_minutes": 14.72,
 "avg_fare_amount": 17.26,
 "avg_total_amount": 27.82
 },
 {
 "day_of_week_num": 3,
 "day_of_week": "Tuesday",
 "time_band": "12-15",
 "trip_count": 115320,
 "avg_duration_minutes": 16.51,
 "avg_fare_amount": 19.01,
 "avg_total_amount": 27.1
 },
 {
 "day_of_week_num": 6,
 "day_of_week": "Friday",
 "time_band": "16-19",
 "trip_count": 113490,
 "avg_duration_minutes": 15.07,
 "avg_fare_amount": 17.62,
 "avg_total_amount": 28.28
 },
 {
 "day_of_week_num": 2,
 "day_of_week": "Monday",
 "time_band": "16-19",
 "trip_count": 108719,
 "avg_duration_minutes": 14.39,
 "avg_fare_amount": 18.8,
 "avg_total_amount": 28.96
 },
 {
 "day_of_week_num": 7,
 "day_of_week": "Saturday",
 "time_band": "16-19",
 "trip_count": 108130,
 "avg_duration_minutes": 15.47,
 "avg_fare_amount": 17.87,
 "avg_total_amount": 25.53
 },
 {
 "day_of_week_num": 1,
 "day_of_week": "Sunday",
 "time_band": "12-15",
 "trip_count": 104864,
 "avg_duration_minutes": 14.61,
 "avg_fare_amount": 19.67,
 "avg_total_amount": 28.09
 },
 {
 "day_of_week_num": 2,
 "day_of_week": "Monday",
 "time_band": "12-15",
 "trip_count": 103251,
 "avg_duration_minutes": 15.16,
 "avg_fare_amount": 19.22,
 "avg_total_amount": 27.5
 },
 {
 "day_of_week_num": 3,
 "day_of_week"

Execution report JSON: /Volumes/db_pruebatecnica/raw/nyc_taxi_files/reports/execution_report_139925ad-ecc9-49a0-8d38-8679529c504c.json


In [0]:
%sql
SHOW CATALOGS;

catalog
nyc_taxi_sebastian
samples
system


In [0]:
%sql
SHOW SCHEMAS IN nyc_taxi_sebastian;

databaseName
default
information_schema
raw
refined
trusted


In [0]:
%sql
SELECT 
  table_schema AS capa,
  table_name AS tabla
FROM nyc_taxi_sebastian.information_schema.tables
WHERE table_schema IN ('raw', 'trusted', 'refined')
ORDER BY table_schema, table_name;

capa,tabla
raw,taxi_zones
raw,yellow_taxi_trips
refined,data_quality_report
refined,execution_report
refined,kpi_borough_economic_efficiency
refined,kpi_data_quality_impact
refined,kpi_temporal_demand
refined,kpi_temporal_peaks
refined,kpi_top10_profitable_zones
refined,kpi_zone_economic_efficiency
